# Earlier-fold training (RB out-of-sample) — Kaggle: GPU T4 x2 + Internet ON

**Run All, walk away (~25 min on GPU).** Trains the folds still needed for the RB weekly out-of-sample
test: through 2020 then 2021 (x seeds 42/43/44). through2019 is already done and is skipped.

WARNING: the setup cell now HARD-FAILS if GPU is not on. Two prior runs stopped after only the fastest
folds — the signature of CPU training (no accelerator). Set Session options -> Accelerator -> GPU T4 x2
before Run All, or the first cell stops with a clear message.

Folds are trained one at a time, all three seeds each, so even a partial run yields WHOLE usable folds:
finishing through2020 alone already gives a 2-of-3-season test (2020 + 2021), which meets the criterion.

Come back to a push or a rb_oos_artifacts.zip. The last cell prints exactly which folds are complete.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"

import torch
print("cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "NO GPU. Set Session options -> Accelerator: GPU T4 x2 (Internet ON), then Run All. "
    "CPU training will not finish all folds.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess

# Config-major (one fold, all 3 seeds) and NEEDED folds first, so a partial
# session still produces whole, usable folds. through2019 is already complete
# and is skipped instantly.
folds = ["configs/transformer_v1_through2020.yaml",   # -> tests 2021
         "configs/transformer_v1_through2021.yaml",   # -> tests 2022
         "configs/transformer_v1_through2019.yaml"]    # already done; skipped
for cfg in folds:
    for seed in (42, 43, 44):
        cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
               "--features-parquet", "data/features_2012_2025.parquet"]
        if seed != 42:
            cmd += ["--seed", str(seed)]
        print()
        print(f"=== {cfg}  seed {seed} ===", flush=True)
        subprocess.run(cmd, check=True)
    print(f"*** {cfg} COMPLETE (all 3 seeds) ***", flush=True)
print()
print("All earlier folds complete.")

In [ ]:
import json
from pathlib import Path
print("which folds are complete (run even after a partial session):")
for year in (2019, 2020, 2021):
    done = [r for r in ("v1", "v1_s43", "v1_s44")
            if (Path(f"models/transformer/{r}/through{year}/metrics.json").exists()
                and json.loads(Path(f"models/transformer/{r}/through{year}/metrics.json").read_text()).get("complete"))]
    print(f"  through{year}: {'READY (3/3)' if len(done)==3 else str(len(done))+'/3'}")
print("READY folds are testable: through2019->2020, through2020->2021, through2021->2022.")

In [ ]:
import subprocess, zipfile
from pathlib import Path
roots = [f"models/transformer/v1{s}" for s in ("", "_s43", "_s44")]
subprocess.run(["git", "add", *roots], check=False)
subprocess.run(["git", "commit", "-m",
                "model: v1 earlier folds for RB out-of-sample test"], check=False)
pushed = subprocess.run(["git", "push"], check=False).returncode == 0
if pushed:
    print("Pushed. Ready for the RB out-of-sample eval.")
else:
    zp = Path("rb_oos_artifacts.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in roots:
            for f in Path(r).rglob("through20*/*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
    print(f"git push failed -- download {zp.resolve()} and drop it in tests/.")